# Kalshi API scratch exploration

Phase 1, Milestone 1: understand and document the live-vs-`/historical/` cutoff distinction, make a manual API call, pretty-print the response, and understand the pagination structure. Findings feed `docs/data-sources.md` and `config.yaml`.


In [1]:
import httpx
import json

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
client = httpx.Client(base_url=BASE_URL, timeout=30.0)


## 1. The live-vs-historical cutoff

Kalshi keeps roughly the most recent 3 months of data in "live" endpoints (`/markets`, `/events`, `/markets/trades`, `/portfolio/...`). Anything older has aged out and can only be reached through a parallel set of `/historical/...` endpoints. `GET /historical/cutoff` returns the exact boundary, as four separate timestamps (one per data type) rather than a single global cutoff.


In [1]:
resp = client.get("/historical/cutoff")
cutoff = resp.json()
print(json.dumps(cutoff, indent=2))


{
  "market_positions_last_updated_ts": "2026-06-06T00:00:00Z",
  "market_settled_ts": "2026-06-06T00:00:00Z",
  "orders_updated_ts": "2026-06-06T00:00:00Z",
  "trades_created_ts": "2026-06-06T00:00:00Z"
}


All four fields land on the same instant here. Reading `market_settled_ts`: any market that **settled before `2026-06-06T00:00:00Z` is only visible through `/historical/markets`** -- the live `/markets` endpoint silently drops it, it won't 404 or error, it just won't be in the response.


### Proving it empirically

`KXCPICOREHEAD` (a top-volume Economics series from the date-range exploration) is a good test case: does it live in the historical archive, the live window, or both?


In [1]:
hist = client.get("/historical/markets", params={"series_ticker": "KXCPICOREHEAD", "limit": 2}).json()
live = client.get("/markets", params={"series_ticker": "KXCPICOREHEAD", "status": "settled", "limit": 2}).json()
print("historical markets found:", len(hist.get("markets", [])))
print("live (settled) markets found:", len(live.get("markets", [])))
if live.get("markets"):
    print("example live settlement_ts:", live["markets"][0].get("settlement_ts"))


historical markets found: 0
live (settled) markets found: 1
example live settlement_ts: 2026-07-14T13:19:09.487219Z


Zero results from `/historical/markets`, one from live `/markets` with a `settlement_ts` of `2026-07-14` -- **after** the cutoff. So `KXCPICOREHEAD` is a series whose settlements are all still in the live window; querying only `/historical/markets` for it (as an earlier version of `test.py` did) silently misses everything. The real ingestion client has to check both endpoints, or pick one based on where the target date range falls relative to the cutoff.


## 2. Pagination structure

Pull one page from `/historical/markets` with no filter, so we're guaranteed a non-empty result to inspect.


In [1]:
resp = client.get("/historical/markets", params={"limit": 2})
page_1 = resp.json()
print(json.dumps(page_1, indent=2))


{
  "cursor": "CgwI07yN0QYQ-PD-xwMSOUtYTVZFU1BPUlRTTVVMVElHQU1FRVhURU5ERUQtUzIwMjYwNkE5ODM0MTNERC1DQThENjMxRjQyQQ",
  "markets": [
    {
      "can_close_early": true,
      "close_time": "2026-06-05T23:45:00Z",
      "created_time": "2026-06-05T23:41:06.122321Z",
      "custom_strike": {
        "Associated Events": "KXBTC15M-26JUN051945,KXNBAGAME-26JUN05NYKSAS",
        "Associated Market Sides": "yes,yes",
        "Associated Markets": "KXBTC15M-26JUN051945-45,KXNBAGAME-26JUN05NYKSAS-SAS",
        "Multivariate Event Ticker": "KXMVESPORTSMULTIGAMEEXTENDED-S202664D882EA4EC"
      },
      "event_ticker": "KXMVESPORTSMULTIGAMEEXTENDED-S202664D882EA4EC",
      "exchange_index": 0,
      "expected_expiration_time": "2026-06-06T03:30:00Z",
      "expiration_time": "2026-06-05T23:45:00Z",
      "expiration_value": "",
      "last_price_dollars": "0.0280",
      "latest_expiration_time": "2026-06-05T23:45:00Z",
      "liquidity_dollars": "0.0000",
      "market_type": "binary",
      "mve_

Two things worth noting in this response:

1. **Unfiltered, this endpoint surfaces a lot of "multivariate event" (MVE) combo markets** -- e.g. a single bet bundling a 15-minute Bitcoin price bin together with an NBA game outcome (`KXMVESPORTSMULTIGAMEEXTENDED-...`). These don't have a clean single `implied_price` the way a normal binary contract does. Worth a note for Phase 2's inclusion-criteria design decision doc -- they likely need explicit exclusion or separate handling.
2. **The pagination shape**: top level is just two keys, `markets` (a list) and `cursor` (a string). That cursor is an opaque, server-generated token -- don't try to parse or construct it, just pass whatever you're given back verbatim.


In [1]:
cursor = page_1["cursor"]
print("cursor:", cursor)

resp2 = client.get("/historical/markets", params={"limit": 2, "cursor": cursor})
page_2 = resp2.json()
print(json.dumps(page_2, indent=2))


cursor: CgwI07yN0QYQ-PD-xwMSOUtYTVZFU1BPUlRTTVVMVElHQU1FRVhURU5ERUQtUzIwMjYwNkE5ODM0MTNERC1DQThENjMxRjQyQQ
{
  "cursor": "CgwI8riN0QYQiPS06wISOUtYTVZFU1BPUlRTTVVMVElHQU1FRVhURU5ERUQtUzIwMjZFOUNBM0JDRTUwQS02Mzg2OEJEODk4RQ",
  "markets": [
    {
      "can_close_early": true,
      "close_time": "2026-06-05T23:45:00Z",
      "created_time": "2026-06-05T23:32:24.533514Z",
      "custom_strike": {
        "Associated Events": "KXBTC15M-26JUN051945,KXETH15M-26JUN051945,KXXRP15M-26JUN051945",
        "Associated Market Sides": "yes,no,yes",
        "Associated Markets": "KXBTC15M-26JUN051945-45,KXETH15M-26JUN051945-45,KXXRP15M-26JUN051945-45",
        "Multivariate Event Ticker": "KXMVESPORTSMULTIGAMEEXTENDED-S202606A983413DD"
      },
      "event_ticker": "KXMVESPORTSMULTIGAMEEXTENDED-S202606A983413DD",
      "exchange_index": 0,
      "expected_expiration_time": "2026-06-05T23:50:00Z",
      "expiration_time": "2026-06-05T23:45:00Z",
      "expiration_value": "",
      "last_price_dollars

Passing `cursor` back in the next request's `cursor` param returns a **different** page -- a new `cursor` value comes back too, ready for a third call. Let's confirm the two pages don't overlap.


In [1]:
tickers_1 = [m["ticker"] for m in page_1["markets"]]
tickers_2 = [m["ticker"] for m in page_2["markets"]]
print("page 1 tickers:", tickers_1)
print("page 2 tickers:", tickers_2)
print("overlap:", set(tickers_1) & set(tickers_2))


page 1 tickers: ['KXMVESPORTSMULTIGAMEEXTENDED-S202664D882EA4EC-53910AFA8B9', 'KXMVESPORTSMULTIGAMEEXTENDED-S202606A983413DD-CA8D631F42A']
page 2 tickers: ['KXMVESPORTSMULTIGAMEEXTENDED-S202606A983413DD-628DA8DE98E', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026E9CA3BCE50A-63868BD898E']
overlap: set()


Empty overlap -- confirmed. `cursor` is genuinely advancing through the dataset, not repeating a page.

### The full loop

A minimal manual version of cursor-based pagination: keep calling with the previous response's `cursor` until the server returns an empty one, which signals the last page. This is the pattern `src/ingest/base.py`'s real `paginate()` generator (Milestone 2) will generalize across every endpoint -- capped here at 5 pages just so this scratch run doesn't pull the entire archive.


In [1]:
def all_pages(limit=100, max_pages=5):
    cursor = None
    page_num = 0
    while page_num < max_pages:
        params = {"limit": limit}
        if cursor:
            params["cursor"] = cursor
        data = client.get("/historical/markets", params=params).json()
        page_num += 1
        markets = data.get("markets", [])
        print(f"page {page_num}: {len(markets)} markets, cursor={data.get('cursor')!r}")
        yield from markets
        cursor = data.get("cursor")
        if not cursor:
            break

all_markets = list(all_pages(limit=50))
print(f"total markets fetched (capped at 5 pages for this demo): {len(all_markets)}")


page 1: 50 markets, cursor='CgwIwamN0QYQ-JKqqAMSL0tYTVZFQ1JPU1NDQVRFR09SWS1TMjAyNkVBMUUxMUEzRjdELThGMUVEMUQzODJG'
page 2: 50 markets, cursor='CgwI-qGN0QYQqMqWiwMSOUtYTVZFU1BPUlRTTVVMVElHQU1FRVhURU5ERUQtUzIwMjY5QzREQ0Q1MEVFMi1GNzk1Q0I0MjI4RQ'
page 3: 50 markets, cursor='CgsIh5uN0QYQoIzeRBIvS1hNVkVDUk9TU0NBVEVHT1JZLVMyMDI2NzkwRTk5NTg4NzMtNkQxRUIxQzJCQ0U'
page 4: 50 markets, cursor='CgwI6Y-N0QYQsNC02AMSOUtYTVZFU1BPUlRTTVVMVElHQU1FRVhURU5ERUQtUzIwMjZENjREQUVGRTVFMi04NjdFMjYzNkQwQg'
page 5: 50 markets, cursor='CgwIzYqN0QYQgOCdtgMSOUtYTVZFU1BPUlRTTVVMVElHQU1FRVhURU5ERUQtUzIwMjY0NUMxOUI3QkE2Qi03MThDRUE4RTU1OA'
total markets fetched (capped at 5 pages for this demo): 250


Each `cursor` is a distinct opaque token and every page returns a full 50 markets (we didn't hit the end of the archive in 5 pages -- there's a lot of historical data). In production, `max_pages` goes away and the loop just runs until `cursor` comes back empty.

## Takeaways for the real ingestion client

- Check `/historical/cutoff` (or just try both) rather than assuming a series' data is entirely live or entirely historical -- `KXCPICOREHEAD` above proved a single series can be 100% on one side of the boundary.
- `cursor` is opaque -- store it as-is for resumable/idempotent ingestion, never try to compute or predict it.
- Unfiltered `/historical/markets` mixes in MVE combo markets that don't fit the `Contract` model's assumption of one clean `implied_price` -- flag for `docs/adr/004-inclusion-criteria.md`.
